# MLP do Zero — Experimentos no MNIST

Notebook de experimentos comparando diferentes configurações do MLP implementado manualmente com NumPy.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from mlp import MLP, load_mnist
from mlp.optimizers import SGD, Adam

## 1. Carregamento e pré-processamento do MNIST

In [ ]:
(X_train_full, y_train_full), (X_test, y_test) = load_mnist('../data')

# Flatten já feito pelo loader; normalização também (0-1)
X_val   = X_train_full[-5000:]
y_val   = y_train_full[-5000:]
X_train = X_train_full[:-5000]
y_train = y_train_full[:-5000]

print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

## 2. Verificação de gradientes (Gradient Check)

Antes de treinar no MNIST, verifica se os gradientes analíticos batem com a aproximação numérica.

In [ ]:
# Rede pequena para o gradient check ser rápido
net_check = MLP([784, 16, 10], hidden_activation='relu', seed=0)
max_diff = net_check.gradient_check(X_train[:10], y_train[:10])
print(f'Max relative difference: {max_diff:.2e}')
assert max_diff < 1e-4, 'Gradient check FAILED — revisar backprop!'
print('Gradient check PASSED ✓')

## 3. Experimento A — Baseline: [784→256→128→10], ReLU, SGD lr=0.1

In [ ]:
opt_a = SGD(learning_rate=0.1)
net_a = MLP([784, 256, 128, 10], hidden_activation='relu', optimizer=opt_a, seed=42)

hist_a = net_a.train(
    X_train, y_train,
    X_val=X_val, y_val=y_val,
    epochs=30, batch_size=128
)

test_acc_a = net_a.accuracy(X_test, y_test)
print(f'\nTest accuracy (Exp A): {test_acc_a:.4f}')

## 4. Experimento B — Mais profundo: [784→512→256→128→10], ReLU, SGD lr=0.1

In [ ]:
opt_b = SGD(learning_rate=0.1)
net_b = MLP([784, 512, 256, 128, 10], hidden_activation='relu', optimizer=opt_b, seed=42)

hist_b = net_b.train(
    X_train, y_train,
    X_val=X_val, y_val=y_val,
    epochs=30, batch_size=128
)

test_acc_b = net_b.accuracy(X_test, y_test)
print(f'\nTest accuracy (Exp B): {test_acc_b:.4f}')

## 5. Experimento C — Adam optimizer: [784→256→128→10], ReLU, Adam lr=0.001

In [ ]:
opt_c = Adam(learning_rate=0.001)
net_c = MLP([784, 256, 128, 10], hidden_activation='relu', optimizer=opt_c, seed=42)

hist_c = net_c.train(
    X_train, y_train,
    X_val=X_val, y_val=y_val,
    epochs=30, batch_size=128
)

test_acc_c = net_c.accuracy(X_test, y_test)
print(f'\nTest accuracy (Exp C): {test_acc_c:.4f}')

## 6. Experimento D — SGD com momentum: [784→256→128→10], ReLU, SGD momentum=0.9

In [ ]:
opt_d = SGD(learning_rate=0.05, momentum=0.9)
net_d = MLP([784, 256, 128, 10], hidden_activation='relu', optimizer=opt_d, seed=42)

hist_d = net_d.train(
    X_train, y_train,
    X_val=X_val, y_val=y_val,
    epochs=30, batch_size=128
)

test_acc_d = net_d.accuracy(X_test, y_test)
print(f'\nTest accuracy (Exp D): {test_acc_d:.4f}')

## 7. Curvas de loss e acurácia

In [ ]:
epochs = range(1, 31)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

configs = [
    ('A — SGD 256-128', hist_a, 'tab:blue'),
    ('B — SGD 512-256-128', hist_b, 'tab:orange'),
    ('C — Adam 256-128', hist_c, 'tab:green'),
    ('D — SGD+mom 256-128', hist_d, 'tab:red'),
]

for label, hist, color in configs:
    axes[0].plot(epochs, hist['val_loss'], label=label, color=color)
    axes[1].plot(epochs, hist['val_acc'], label=label, color=color)

axes[0].set_title('Validation Loss por Época')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Validation Accuracy por Época')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Acurácia')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/curves_comparison.png', bbox_inches='tight')
plt.show()
print('Figura salva em results/curves_comparison.png')

## 8. Tabela comparativa

In [ ]:
results = [
    ('A', '256-128',     'ReLU', 'SGD lr=0.1',          test_acc_a),
    ('B', '512-256-128', 'ReLU', 'SGD lr=0.1',          test_acc_b),
    ('C', '256-128',     'ReLU', 'Adam lr=0.001',        test_acc_c),
    ('D', '256-128',     'ReLU', 'SGD mom=0.9 lr=0.05',  test_acc_d),
]

print(f'{"Exp":<6} {"Camadas":<16} {"Ativacao":<10} {"Otimizador":<24} {"Test Acc"}')
print('-' * 72)
for exp, layers, act, opt, acc in results:
    print(f'{exp:<6} {layers:<16} {act:<10} {opt:<24} {acc:.4f}')

## 9. Matriz de confusão (melhor modelo)

In [ ]:
# Seleciona o modelo com maior test accuracy
best_net = max(
    [(net_a, test_acc_a), (net_b, test_acc_b), (net_c, test_acc_c), (net_d, test_acc_d)],
    key=lambda x: x[1]
)[0]

y_pred_test = best_net.predict(X_test)

# Confusion matrix manual
n_classes = 10
cm = np.zeros((n_classes, n_classes), dtype=int)
for true, pred in zip(y_test, y_pred_test):
    cm[true, pred] += 1

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im, ax=ax)

ax.set_xticks(range(n_classes))
ax.set_yticks(range(n_classes))
ax.set_xlabel('Predito')
ax.set_ylabel('Real')
ax.set_title('Matriz de Confusão — Melhor Modelo')

for i in range(n_classes):
    for j in range(n_classes):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', color=color, fontsize=8)

plt.tight_layout()
plt.savefig('../results/confusion_matrix.png', bbox_inches='tight')
plt.show()

# Erros mais comuns
print('\nPares mais confundidos (real → predito):')
off_diag = [(cm[i, j], i, j) for i in range(n_classes) for j in range(n_classes) if i != j]
for count, real, pred in sorted(off_diag, reverse=True)[:5]:
    print(f'  {real} → {pred}: {count} erros')

## 10. Visualização das ativações da primeira camada (pesos W1)

In [ ]:
# Visualiza os primeiros 64 filtros aprendidos pela camada W1
W1 = best_net.params['W1']  # shape (784, n_hidden)
n_show = min(64, W1.shape[1])

fig, axes = plt.subplots(8, 8, figsize=(10, 10))
for idx, ax in enumerate(axes.flat):
    if idx < n_show:
        neuron_weights = W1[:, idx].reshape(28, 28)
        ax.imshow(neuron_weights, cmap='RdBu', vmin=-0.3, vmax=0.3)
    ax.axis('off')

plt.suptitle('Pesos aprendidos — camada oculta 1 (64 neurônios)', y=1.01)
plt.tight_layout()
plt.savefig('../results/weights_visualization.png', bbox_inches='tight')
plt.show()

## 11. t-SNE dos embeddings da penúltima camada

In [ ]:
from sklearn.manifold import TSNE

# Extrai ativacoes da penultima camada para 2000 exemplos do teste
X_subset = X_test[:2000]
y_subset = y_test[:2000]

# Forward parcial ate a penultima camada
A = X_subset
for i in range(best_net.n_layers - 1):
    Z = A @ best_net.params[f'W{i+1}'] + best_net.params[f'b{i+1}']
    A = best_net.act_fn(Z)
embeddings = A

print('Rodando t-SNE (pode demorar ~30s)...')
tsne = TSNE(n_components=2, random_state=42, perplexity=40, max_iter=1000)
emb_2d = tsne.fit_transform(embeddings)

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(emb_2d[:, 0], emb_2d[:, 1], c=y_subset, cmap='tab10', s=5, alpha=0.7)
plt.colorbar(scatter, ax=ax, label='Digito')
ax.set_title('t-SNE - Embeddings da penultima camada (2000 exemplos de teste)')
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
plt.tight_layout()
plt.savefig('../results/tsne_embeddings.png', bbox_inches='tight')
plt.show()